# Large Language Models

The following tutorial is a summary of Andrej Karpathy's video [A Deep Dive into Large Language Models like ChatGPT](https://www.youtube.com/watch?v=Te5rOTcE3FQ). I highly recommend watching the video, as it contains a lot of useful information and visualizations that are hard to capture in text.

Why should you read this?

* You want to understand how LLMs actually work not just at the surface level.

* You want to understand confusing fine-tuning terms like chat_template and ChatML (especially if you're using Axolotl).

* You want to get better at prompt engineering by understanding why some prompts work better than others.

* You’re trying to reduce hallucinations and want to know how to keep LLMs from making things up.

* You want to understand why DeepSeek-R1 is such a big deal right now.

Note: If you are looking for the excalidraw diagram that Andrej made for the video, you can download it [here](https://cdn.anfalmushtaq.com/static/imgs/blog/2025-02-08-deep-dive-into-llms-like-chatgpt-tldr/LLMIntro-2024-07-22-1743.excalidraw). He shared it through Google Drive and it invalidates the link after a certain time. That's why I have decided to host it on my CDN as well.

## Preview of Things to Come

Future LLMs will expand in several key areas:

* **Multimodal Capabilities** → Not just text, but also understanding and generating images, audio, and video.
* **Agent-Based Models** → Moving beyond single tasks to long-term memory, reasoning, and correction of mistakes.
* **Pervasive & Invisible AI** → AI will be integrated into workflows in a way that becomes second nature.
* **Computer-Using AI** → AI models that interact with software and take actions beyond just text generation.
* **Test-Time Training** → AI adapting itself in real-time to improve accuracy on the fly.

## Keeping Track of LLMs

If you're interested in following developments in this space, here are some great resources:

* [LM Arena](https://lmarena.ai/) → Benchmarking new language models.
* [AI News](https://news.smol.ai/) → A newsletter covering AI research.

## Where to Find LLMs

Want to try different LLMs? Here’s where to find them:

* **Proprietary Models** → OpenAI (GPT-4), Google (Gemini), Anthropic (Claude), etc.
* **Open-Weight Models** → DeepSeek, Meta (Llama), etc. Try them via Together.ai.
* **Run Locally** → Use [Ollama](https://ollama.com/) or [LM Studio](https://lmstudio.ai/).
* **Base Models** → Explore [Hyperbolic](https://app.hyperbolic.xyz/).


## Tokenization 

A text processing pipeline begins with a tokenizer. This splits the text into smaller
constituent units (tokens) from a vocabulary of possible tokens. In the discussion above,
we have implied that these tokens represent words, but there are several difficulties.

* Inevitably, some words (e.g., names) will not be in the vocabulary.

* It’s unclear how to handle punctuation, but this is important. If a sentence ends
in a question mark, we must encode this information.

* The vocabulary would need different tokens for versions of the same word with
different suffixes (e.g., walk, walks, walked, walking), and there is no way to clarify
that these variations are related.

One approach would be to use letters and punctuation marks as the vocabulary, but this
would mean splitting text into very small parts and requiring the subsequent network to
re-learn the relations between them.

In practice, a compromise between letters and full words is used, and the final vocabulary includes both common words and word fragments from which larger and less
frequent words can be composed. The vocabulary is computed using a sub-word tokenizer such as <mark>Byte Pair Encoding (BPE)</mark> that greedily merges commonly occurring sub-strings based on their frequency.

<img src="assets/Chap12/TransformerBPE.svg" style="filter: invert(1);" width="100%">

Sub-word tokenization. 

a) A passage of text from a nursery rhyme.
The tokens are initially just the characters and whitespace (represented by an
underscore), and their frequencies are displayed in the table. 

b) At each iteration,
the sub-word tokenizer looks for the most commonly occurring adjacent pair of
tokens (in this case, se) and merges them. This creates a new token and decreases
the counts for the original tokens s and e. Note that the last character of the first
token to be merged cannot be whitespace, which prevents merging across words.

c) At the second iteration, the algorithm merges e and the whitespace character_.

d) After 23 iterations, the tokens consist of a mix of letters, word fragments, and
commonly occurring words. 

e) If we continue this process indefinitely, the tokens
eventually represent the full words. 

f) Over time, the number of tokens increases
as we add word fragments to the letters and then decreases again as we merge
these fragments. In a real situation, there would be a very large number of words,
and the algorithm would terminate when the vocabulary size (number of tokens)
reached a predetermined value. Punctuation and capital letters would also be
treated as separate input characters.


You can visualize how this works using tools like [Tiktokenizer (https://tiktokenizer.vercel.app)](https://tiktokenizer.vercel.app/?model=cl100k_base).

For reference, GPT-3 uses 50,257 tokens and GPT-4 uses 100,277 tokens in its vocabulary.

## Embeddings 

Each token in the vocabulary $\mathcal{V}$ is mapped to a unique word embedding, and the embeddings for the whole vocabulary are stored in a matrix $\Omega_e \in \mathbb{R}^{D \times |\mathcal{V}|}$. 

To accomplish this, the $N$ input tokens are first encoded in the matrix $T \in \mathbb{R}^{|\mathcal{V}| \times N}$, where the nth column
corresponds to the nth token and is a |V|×1 one-hot vector (i.e., a vector where every
entry is zero except for the entry corresponding to the token, which is set to one). The
input embeddings are computed as $X= \Omega_e T$, and $\Omega_e$ is learned like any other network
parameter. A typical embedding size D is 1024, and a typical total vocabulary size $|\mathcal{V}|$ is 30,000, so even before the main network, there are many parameters
in $\Omega_e$ to learn.

<img src="assets/Chap12/TransformerVocabulary.svg" style="filter: invert(1);" width="100%">

The input embedding matrix $\mathbf{X} \in \mathbb{R}^{D \times N}$ contains N embeddings of length D and is created by multiplying a matrix $\mathbf{\Omega_e}$ containing the embeddings that correspond to the word or sub-word indices. The vocabulary matrix $\mathbf{\Omega_e}$ is considered a parameter of the model and is learned along with the other parameters. Note that the two embeddings for the word an in $\mathbf{X}$ are the same.
